# Kronos Trading System - Jupyter Tutorial

**Version**: v5.0.0  
**Last Updated**: 2026-04-27  

---

## Table of Contents

1. [Introduction](#1-introduction)
2. [System Architecture](#2-system-architecture)
3. [Getting Started](#3-getting-started)
4. [Core Components](#4-core-components)
5. [Trading Strategies](#5-trading-strategies)
6. [Risk Management](#6-risk-management)
7. [Backtesting](#7-backtesting)
8. [Live Monitoring](#8-live-monitoring)
9. [API Reference](#9-api-reference)
10. [Troubleshooting](#10-troubleshooting)

## 1. Introduction

**Kronos** is an autonomous cryptocurrency trading system built with a 5-layer architecture.

### Key Features:
- **5-Layer Architecture**: Core → Strategies → Models → Risk → Data
- **Multi-Timeframe Analysis**: 1H, 4H, 1D timeframes
- **Regime Detection**: BULL, BEAR, RANGE, VOLATILE market detection
- **Circuit Breaker**: Automatic trading halt on consecutive losses
- **Paper Trading**: Simulation mode for safe testing

In [ ]:
# Example: Import core components
import sys
sys.path.insert(0, '/Users/jimingzhang/kronos')

try:
    from core.constants import SYSTEM_VERSION, RiskConfig, MarketRegime, SignalType
    print(f"✓ Kronos v{SYSTEM_VERSION} loaded successfully")
    print(f"  - RiskConfig: hourly_loss_pct={RiskConfig.hourly_loss_pct}")
    print(f"  - MarketRegime options: {[r.name for r in MarketRegime]}")
    print(f"  - SignalType options: {[s.name for s in SignalType]}")
except ImportError as e:
    print(f"✗ Import error: {e}")
    print("  Make sure you're running from the kronos directory or it's in your PYTHONPATH")

## 2. System Architecture

```
┌─────────────────────────────────────────────────────────────────┐
│                    KRONOS v5.0 ARCHITECTURE                    │
├─────────────────────────────────────────────────────────────────┤
│  Entry Points          │  5-Layer Pipeline                      │
│  ─────────────────────┼──────────────────────────────────────  │
│  • kronos_pilot.py     │  P0: Core (constants, config,          │
│  • kronos_auto_guard   │       indicators, gemma4_parser)        │
│  • kronos_heartbeat    │  P1: Strategies (regime, alpha, beta)   │
│  • real_monitor.py     │  P2: Models (confidence, position)      │
│                        │  P3: Risk (circuit breaker, trailing)  │
│                        │  P4: Data (ATR watchlist, evolution)   │
│                        │  P5: Execution (reserved)               │
└─────────────────────────────────────────────────────────────────┘
```

## 3. Getting Started

### 3.1 Environment Setup

```bash
# Install dependencies
cd /Users/jimingzhang/kronos
pip install -r requirements.txt

# Set environment variables
export OKX_API_KEY="your_api_key"
export OKX_SECRET="your_secret"
export OKX_FLAG="1"  # 1=simulation, 0=live
```

In [ ]:
# 3.2 Configuration Check
import os

# Check OKX configuration
OKX_FLAG = os.getenv('OKX_FLAG', '1')
DEMO_MODE = OKX_FLAG == '1'

print(f"Trading Mode: {'SIMULATION (Paper Trading)' if DEMO_MODE else 'LIVE TRADING'}")
print(f"OKX_FLAG={OKX_FLAG}")

if not DEMO_MODE:
    print("\n⚠️  WARNING: Live trading is enabled! Ensure you know what you're doing.")

## 4. Core Components

### 4.1 Technical Indicators

The system calculates various technical indicators for trading decisions.

In [ ]:
# Example: Using indicators
try:
    from core.indicators import (
        calculate_rsi, calculate_adx, calculate_atr,
        calculate_bollinger_bands, calculate_macd
    )
    
    import numpy as np
    
    # Sample price data
    np.random.seed(42)
    prices = 100 + np.cumsum(np.random.randn(100) * 2)
    
    # Calculate indicators
    rsi = calculate_rsi(prices)
    adx = calculate_adx(prices)
    atr = calculate_atr(prices)
    
    print("✓ Technical Indicators Calculated:")
    print(f"  - RSI(14): {rsi[-1]:.2f}")
    print(f"  - ADX(14): {adx[-1]:.2f}")
    print(f"  - ATR(14): {atr[-1]:.4f}")
    
    # Interpret RSI
    if rsi[-1] < 30:
        print(f"  → RSI indicates OVERSOLD condition (bullish potential)")
    elif rsi[-1] > 70:
        print(f"  → RSI indicates OVERBOUGHT condition (bearish potential)")
    else:
        print(f"  → RSI indicates NEUTRAL condition")
        
except ImportError as e:
    print(f"Note: Could not load indicators module: {e}")
    print("  This is expected if running outside the kronos environment.")

### 4.2 Market Regime Classification

The regime classifier determines the current market state:

In [ ]:
# Example: Market Regime Types
try:
    from core.constants import MarketRegime
    
    print("Market Regime Types:")
    for regime in MarketRegime:
        print(f"  • {regime.name}: {regime.value}")
        
except ImportError as e:
    print(f"Could not load MarketRegime: {e}")

## 5. Trading Strategies

### 5.1 Alpha Engine (Mean Reversion)

The Alpha Engine generates signals based on mean reversion principles.

In [ ]:
# Example: Signal Types
try:
    from core.constants import SignalType
    
    print("Signal Types:")
    for sig in SignalType:
        print(f"  • {sig.name}: {sig.value}")
        
except ImportError as e:
    print(f"Could not load SignalType: {e}")

### 5.2 Beta Engine (Trend Following)

The Beta Engine follows trends using multiple timeframe analysis.

In [ ]:
# Multi-Timeframe Analysis Example
timeframes = {
    '1h': '1-hour chart',
    '4h': '4-hour chart', 
    '1d': '1-day chart'
}

print("Multi-Timeframe Analysis:")
for tf, desc in timeframes.items():
    print(f"  • {tf}: {desc}")

print("\nSignal Confirmation Rules:")
print("  • For BULL signals: 1h trend should align with 4h and 1d")
print("  • For BEAR signals: 1h trend should align with 4h and 1d")
print("  • No trade if timeframes disagree")

## 6. Risk Management

### 6.1 Circuit Breaker

The circuit breaker prevents cascading losses by halting trading after consecutive failures.

In [ ]:
# Example: Circuit Breaker Configuration
try:
    from core.constants import RiskConfig
    
    print("Risk Management Configuration:")
    print(f"  • Hourly Loss Limit: {RiskConfig.hourly_loss_pct*100}%")
    print(f"  • Daily Loss Limit: {RiskConfig.daily_loss_pct*100}%")
    print(f"  • Per-Trade Loss Limit: {RiskConfig.per_trade_pct*100}%")
    print(f"  • Reserve Buffer: {RiskConfig.reserve_pct*100}%")
    
except ImportError as e:
    print(f"Could not load RiskConfig: {e}")

### 6.2 Position Sizing

Position sizes are calculated based on confidence scores and risk parameters.

In [ ]:
# Example: Position Sizing Calculation
def calculate_position_size(account_balance, risk_pct, entry_price, stop_loss_pct):
    """
    Calculate position size based on risk management rules.
    
    Args:
        account_balance: Total account value
        risk_pct: Risk percentage per trade (e.g., 0.01 for 1%)
        entry_price: Entry price for the trade
        stop_loss_pct: Stop loss percentage (e.g., 0.02 for 2%)
    
    Returns:
        dict with position size and risk amount
    """
    risk_amount = account_balance * risk_pct
    position_size = risk_amount / stop_loss_pct
    
    return {
        'position_size': position_size,
        'risk_amount': risk_amount,
        'notional_value': position_size * entry_price,
        'stop_loss_price': entry_price * (1 - stop_loss_pct)
    }

# Example calculation
result = calculate_position_size(
    account_balance=10000,
    risk_pct=0.01,  # 1% risk
    entry_price=100,
    stop_loss_pct=0.02  # 2% stop loss
)

print("Position Sizing Example:")
print(f"  • Account Balance: ${result['risk_amount']/0.01:,.2f}")
print(f"  • Risk Amount (1%): ${result['risk_amount']:,.2f}")
print(f"  • Position Size: {result['position_size']:.4f} units")
print(f"  • Notional Value: ${result['notional_value']:,.2f}")
print(f"  • Stop Loss Price: ${result['stop_loss_price']:.2f}")

## 7. Backtesting

### 7.1 Running a Backtest

The backtest engine allows you to test strategies against historical data.

In [ ]:
# Example: Backtest Results Structure
backtest_result = {
    'symbol': 'BTC/USDT',
    'period': '2024-01-01 to 2024-12-31',
    'total_trades': 150,
    'win_rate': 0.58,
    'total_pnl_pct': 24.5,
    'max_drawdown_pct': 8.2,
    'sharpe_ratio': 1.45,
    'avg_trade_duration_hours': 18.5
}

print("Backtest Results Example:")
print(f"  • Symbol: {backtest_result['symbol']}")
print(f"  • Period: {backtest_result['period']}")
print(f"  • Total Trades: {backtest_result['total_trades']}")
print(f"  • Win Rate: {backtest_result['win_rate']*100:.1f}%")
print(f"  • Total PnL: {backtest_result['total_pnl_pct']:.1f}%")
print(f"  • Max Drawdown: {backtest_result['max_drawdown_pct']:.1f}%")
print(f"  • Sharpe Ratio: {backtest_result['sharpe_ratio']:.2f}")

### 7.2 Walk-Forward Optimization

Walk-forward analysis validates strategy robustness by testing on out-of-sample data.

In [ ]:
# Example: Walk-Forward Results
wf_results = {
    'in_sample': {'sharpe': 1.8, 'win_rate': 0.62},
    'out_of_sample': {'sharpe': 1.5, 'win_rate': 0.58},
    'rollage_ratio': 0.85  # out_of_sample / in_sample
}

print("Walk-Forward Analysis:")
print(f"  • In-Sample Sharpe: {wf_results['in_sample']['sharpe']:.2f}")
print(f"  • Out-of-Sample Sharpe: {wf_results['out_of_sample']['sharpe']:.2f}")
print(f"  • Rollage Ratio: {wf_results['rollage_ratio']:.2%}")

if wf_results['rollage_ratio'] > 0.7:
    print("  → Strategy is ROBUST (rollage > 70%)")
else:
    print("  → Strategy may be OVERFITTING (rollage < 70%)")

## 8. Live Monitoring

### 8.1 System Status Check

Check the current system status and active positions.

In [ ]:
# Example: System Status
system_status = {
    'mode': 'SIMULATION',
    'circuit_breaker': 'ACTIVE',
    'consecutive_losses': 0,
    'active_positions': [
        {'symbol': 'BTC/USDT', 'side': 'LONG', 'pnl_pct': 1.5},
        {'symbol': 'ETH/USDT', 'side': 'SHORT', 'pnl_pct': -0.3}
    ],
    'daily_pnl_pct': 0.8,
    'last_signal_time': '2026-04-27T20:00:00Z'
}

print("System Status:")
print(f"  • Mode: {system_status['mode']}")
print(f"  • Circuit Breaker: {system_status['circuit_breaker']}")
print(f"  • Consecutive Losses: {system_status['consecutive_losses']}")
print(f"  • Daily PnL: {system_status['daily_pnl_pct']:.1f}%")
print(f"\nActive Positions:")
for pos in system_status['active_positions']:
    emoji = "📈" if pos['pnl_pct'] > 0 else "📉"
    print(f"  {emoji} {pos['symbol']}: {pos['side']} | PnL: {pos['pnl_pct']:+.1f}%")

### 8.2 Feishu Notifications

The system sends alerts to Feishu for important events.

In [ ]:
# Example: Notification Types
notification_types = {
    'trade_executed': '📊 Trade Executed',
    'circuit_breaker_triggered': '🔴 Circuit Breaker Triggered',
    'position_closed': '✅ Position Closed',
    'daily_summary': '📈 Daily Summary',
    'emergency_stop': '🛑 EMERGENCY STOP',
    'weekly_review': '📋 Weekly Review'
}

print("Feishu Notification Types:")
for key, desc in notification_types.items():
    print(f"  • {key}: {desc}")

## 9. API Reference

### Key Functions

| Module | Function | Description |
|--------|----------|-------------|
| `core.engine` | `KronosEngine` | Main trading engine |
| `core.indicators` | `calculate_rsi()` | Calculate RSI indicator |
| `core.indicators` | `calculate_adx()` | Calculate ADX indicator |
| `core.indicators` | `calculate_atr()` | Calculate ATR indicator |
| `strategies.regime` | `RegimeClassifier` | Classify market regime |
| `strategies.alpha` | `AlphaEngine` | Generate alpha signals |
| `risk.circuit` | `CircuitBreaker` | Circuit breaker logic |
| `data.atr` | `ATRWatchlist` | Track volatility watchlist |

### 9.1 Engine Usage

```python
# Basic engine usage
from core.engine import KronosEngine

engine = KronosEngine()
signal = engine.generate_signal('BTC/USDT', '1h')
if signal:
    print(f"Signal: {signal.type}, Confidence: {signal.confidence}")
```

In [ ]:
# Example: Signal Structure
signal_example = {
    'type': 'LONG',  # LONG, SHORT, CLOSE
    'confidence': 0.75,
    'regime': 'BULL',
    'entry_price': 95000,
    'stop_loss': 93000,
    'take_profit': 99000,
    'position_size_pct': 20,  # % of account
    'timeframe': '1h',
    'reasons': [
        'RSI oversold',
        '4h trend bullish',
        'High confidence score'
    ]
}

print("Signal Structure Example:")
for key, value in signal_example.items():
    print(f"  • {key}: {value}")

## 10. Troubleshooting

### Common Issues

#### 1. Import Errors
```python
import sys
sys.path.insert(0, '/Users/jimingzhang/kronos')
```

#### 2. API Key Issues
```bash
export OKX_API_KEY="your_key"
export OKX_SECRET="your_secret"
```

#### 3. Circuit Breaker Triggered
Check `data/circuit.json` for consecutive loss count. Reset after review.

#### 4. Data Fetching Issues
Ensure `OKX_FLAG="1"` for simulation mode to use simulated data.

In [ ]:
# Diagnostic Helper
def run_diagnostics():
    """Run basic system diagnostics."""
    
    print("=" * 50)
    print("KRONOS SYSTEM DIAGNOSTICS")
    print("=" * 50)
    
    # 1. Version Check
    try:
        from core.constants import SYSTEM_VERSION
        print(f"✓ Version: v{SYSTEM_VERSION}")
    except:
        print("✗ Could not load SYSTEM_VERSION")
    
    # 2. Python Version
    import sys
    print(f"✓ Python: {sys.version.split()[0]}")
    
    # 3. Path Check
    import os
    kronos_path = '/Users/jimingzhang/kronos'
    if os.path.exists(kronos_path):
        print(f"✓ Kronos path exists: {kronos_path}")
    else:
        print(f"✗ Kronos path not found: {kronos_path}")
    
    # 4. Trading Mode
    flag = os.getenv('OKX_FLAG', '1')
    print(f"✓ OKX_FLAG: {flag} ({'SIMULATION' if flag == '1' else 'LIVE'})")
    
    # 5. Required Modules
    required = ['numpy', 'pandas', 'requests', 'okx']
    for mod in required:
        try:
            __import__(mod)
            print(f"✓ Module: {mod}")
        except ImportError:
            print(f"✗ Missing: {mod}")
    
    print("=" * 50)
    print("Diagnostics Complete")
    print("=" * 50)

run_diagnostics()

---

## Summary

This tutorial covered:

1. **System Architecture**: 5-layer design from Core to Execution
2. **Core Components**: Indicators, regime classification, signal types
3. **Trading Strategies**: Alpha (mean reversion) and Beta (trend following)
4. **Risk Management**: Circuit breaker, position sizing, loss limits
5. **Backtesting**: Walk-forward optimization and performance metrics
6. **Live Monitoring**: System status and notifications

For more information, see:
- `README.md` - Full system documentation
- `CLAUDE.md` - AI assistant guide
- `docs/` - Detailed documentation